In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    lit,
    when,
    concat,
    avg,
    count,
    min,
    max
)

# ============================================================
# 1. CREATE SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("DataFrameTransformationsActions") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created")


# ============================================================
# 2. LOAD DATA
# ============================================================

employees = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/employees.csv")

departments = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/departments.csv")


# ============================================================
# 3. ACTION - show()
# ============================================================

print("\n=== ORIGINAL EMPLOYEE DATA ===")

employees.show()


# ============================================================
# 4. NARROW TRANSFORMATION - filter()
# ============================================================

print("\n=== HIGH SALARY EMPLOYEES ===")

high_salary = employees.filter(
    col("salary") > 100000
)

high_salary.show()


# ============================================================
# 5. NARROW TRANSFORMATION - select()
# ============================================================

print("\n=== SELECTED COLUMNS ===")

selected = employees.select(
    "name",
    "job_title",
    "salary"
)

selected.show()


# ============================================================
# 6. NARROW TRANSFORMATION - withColumn()
# ============================================================

print("\n=== ANNUAL SALARY ===")

annual_salary = employees.withColumn(
    "annual_salary",
    col("salary") * 12
)

annual_salary.select(
    "name",
    "salary",
    "annual_salary"
).show()


# ============================================================
# 7. NARROW TRANSFORMATION - drop()
# ============================================================

print("\n=== DROP COLUMNS ===")

dropped = employees.drop(
    "age",
    "city"
)

dropped.show()


# ============================================================
# 8. COLUMN FUNCTION - col()
# ============================================================

print("\n=== USING col() ===")

employees.filter(
    col("salary") > 90000
).select(
    col("name"),
    col("salary")
).show()


# ============================================================
# 9. COLUMN FUNCTION - lit()
# ============================================================

print("\n=== USING lit() ===")

company_df = employees.withColumn(
    "company",
    lit("ABC Technologies")
)

company_df.select(
    "name",
    "company"
).show()


# ============================================================
# 10. COLUMN FUNCTION - when()
# ============================================================

print("\n=== USING when() ===")

salary_category = employees.withColumn(
    "salary_level",
    when(col("salary") >= 150000, "Very High")
    .when(col("salary") >= 100000, "High")
    .when(col("salary") >= 70000, "Medium")
    .otherwise("Low")
)

salary_category.select(
    "name",
    "salary",
    "salary_level"
).show()


# ============================================================
# 11. COLUMN FUNCTION - concat()
# ============================================================

print("\n=== USING concat() ===")

employee_label = employees.withColumn(
    "employee_label",
    concat(
        col("name"),
        lit(" - "),
        col("job_title")
    )
)

employee_label.select(
    "employee_id",
    "employee_label"
).show(truncate=False)


# ============================================================
# 12. WIDE TRANSFORMATION - groupBy()
# ============================================================

print("\n=== GROUP BY DEPARTMENT ===")

department_stats = employees.groupBy(
    "department_id"
).agg(
    count("*").alias("employee_count"),
    avg("salary").alias("average_salary"),
    min("salary").alias("minimum_salary"),
    max("salary").alias("maximum_salary")
)

department_stats.show()


# ============================================================
# 13. WIDE TRANSFORMATION - join()
# ============================================================

print("\n=== JOIN EMPLOYEES AND DEPARTMENTS ===")

employee_details = employees.join(
    departments,
    "department_id",
    "inner"
)

employee_details.select(
    "employee_id",
    "name",
    "job_title",
    "salary",
    "department_name",
    "location"
).show()


# ============================================================
# 14. WIDE TRANSFORMATION - distinct()
# ============================================================

print("\n=== UNIQUE CITIES ===")

employees.select(
    "city"
).distinct().show()


# ============================================================
# 15. ACTION - count()
# ============================================================

print("\n=== COUNT ===")

print(
    "Total Employees:",
    employees.count()
)


# ============================================================
# 16. ACTION - take()
# ============================================================

print("\n=== TAKE 5 ===")

rows = employees.take(5)

for row in rows:
    print(
        row.employee_id,
        row.name,
        row.salary
    )


# ============================================================
# 17. ACTION - collect()
# ============================================================

print("\n=== COLLECT ===")

rows = employees.collect()

for row in rows:
    print(row.name, row.salary)


# ============================================================
# 18. STOP SPARK
# ============================================================

spark.stop()

print("\nSpark Session Stopped")

Spark Session Created

=== ORIGINAL EMPLOYEE DATA ===
+-----------+------------+---+-------------+--------------------+------+---------+
|employee_id|        name|age|department_id|           job_title|salary|     city|
+-----------+------------+---+-------------+--------------------+------+---------+
|        101| Amit Sharma| 28|           10|   Software Engineer| 65000| Vadodara|
|        102|  Neha Patel| 32|           20|          HR Manager| 85000|Ahmedabad|
|        103|   Raj Mehta| 35|           10|Senior Software E...|105000|     Pune|
|        104|  Priya Shah| 26|           30|          Accountant| 55000|   Mumbai|
|        105| Rohan Desai| 41|           10|           Architect|145000|Bengaluru|
|        106|Anjali Joshi| 30|           20|        HR Executive| 60000| Vadodara|
|        107|  Vivek Shah| 38|           30|     Finance Manager|120000|Ahmedabad|
|        108| Pooja Mehta| 25|           10|    Junior Developer| 45000|     Pune|
|        109| Karan Patel| 45|   